In [4]:
import pandas as pd

df = pd.read_csv("train.csv")
cols = df.drop(columns=["is_claim"]).columns
print(cols)

Index(['policy_id', 'policy_tenure', 'age_of_car', 'age_of_policyholder',
       'area_cluster', 'population_density', 'make', 'segment', 'model',
       'fuel_type', 'max_torque', 'max_power', 'engine_type', 'airbags',
       'is_esc', 'is_adjustable_steering', 'is_tpms', 'is_parking_sensors',
       'is_parking_camera', 'rear_brakes_type', 'displacement', 'cylinder',
       'transmission_type', 'gear_box', 'steering_type', 'turning_radius',
       'length', 'width', 'height', 'gross_weight', 'is_front_fog_lights',
       'is_rear_window_wiper', 'is_rear_window_washer',
       'is_rear_window_defogger', 'is_brake_assist', 'is_power_door_locks',
       'is_central_locking', 'is_power_steering',
       'is_driver_seat_height_adjustable', 'is_day_night_rear_view_mirror',
       'is_ecw', 'is_speed_alert', 'ncap_rating'],
      dtype='object')


In [1]:
import joblib
import pandas as pd
import numpy as np

# -------------------------
# LOAD MODEL
# -------------------------
model = joblib.load("car_claim_numeric_model.pkl")

# -------------------------
# LOAD TRAIN STRUCTURE
# -------------------------
train_df = pd.read_csv("train.csv")

# -------------------------
# CREATE INPUT TEMPLATE
# -------------------------
input_data = train_df.drop(columns=["is_claim"]).iloc[0:1].copy()

# -------------------------
# MODIFY VALUES (USER INPUT)
# -------------------------
input_data["age_of_policyholder"] = 35
input_data["age_of_car"] = 2
input_data["fuel_type"] = "Petrol"
input_data["airbags"] = 4

# -------------------------
# 🔥 SAME CLEANING AS TRAINING
# -------------------------
def extract_number(x):
    try:
        return float(str(x).split()[0])
    except:
        return np.nan

for col in ["max_power", "gross_weight"]:
    if col in input_data.columns:
        input_data[col] = input_data[col].apply(extract_number)
        input_data[col] = pd.to_numeric(input_data[col], errors="coerce")

# -------------------------
# 🔥 SAME FEATURE ENGINEERING
# -------------------------
if "max_power" in input_data.columns and "gross_weight" in input_data.columns:
    input_data["engine_per_weight"] = input_data["max_power"] / (input_data["gross_weight"] + 1)

if "age_of_car" in input_data.columns and "age_of_policyholder" in input_data.columns:
    input_data["age_ratio"] = input_data["age_of_car"] / (input_data["age_of_policyholder"] + 1)

# -------------------------
# PREDICT
# -------------------------
prediction = model.predict(input_data)[0]
claim_prob = model.predict_proba(input_data)[:,1][0]

# -------------------------
# RISK LOGIC (SMART OUTPUT)
# -------------------------
if claim_prob < 0.30:
    risk_level = "Low Risk"
    approval_likelihood = "High"
    decision = "Auto Approve"
elif claim_prob < 0.60:
    risk_level = "Medium Risk"
    approval_likelihood = "Moderate"
    decision = "Manual Review"
else:
    risk_level = "High Risk"
    approval_likelihood = "Low"
    decision = "Investigate"

# -------------------------
# OUTPUT
# -------------------------
print("\n" + "="*40)
print("        INSURANCE CLAIM RESULT")
print("="*40)

print(f"\n🔍 Claim Approval Chance: {100 - (claim_prob*100):.2f}%")
print(f"📊 Claim Risk Probability: {claim_prob*100:.2f}%")

print("\n📌 Risk Category:")
print(f"   → {risk_level}")

print("\n📈 Approval Chances:")
print(f"   → {approval_likelihood}")

print("\n🧾 Final Decision:")
print(f"   → {decision}")

# Simple explanation for customer
print("\n💡 Explanation:")
if claim_prob < 0.30:
    print("   Your profile looks safe. Claim is likely to be approved quickly.")
elif claim_prob < 0.60:
    print("   Some risk detected. Your claim may require manual verification.")
else:
    print("   High risk detected. Claim may be delayed or investigated.")

print("="*40)

C:\Users\acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['max_power' 'engine_per_weight']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['max_power' 'engine_per_weight']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(



        INSURANCE CLAIM RESULT

🔍 Claim Approval Chance: 81.30%
📊 Claim Risk Probability: 18.70%

📌 Risk Category:
   → Low Risk

📈 Approval Chances:
   → High

🧾 Final Decision:
   → Auto Approve

💡 Explanation:
   Your profile looks safe. Claim is likely to be approved quickly.
